# Data Preparation
## Part 1: Prepare SATCAT Data
#### 1. Acquire all debris data from SATCAT via API from [space-track.org](https://www.space-track.org/) using the [spacetrack module](https://spacetrack.readthedocs.io/en/stable/)

In [2]:
# Space-Track.org API authentication 
from spacetrack import SpaceTrackClient

# read Space-Track.org credentials from external file (username in line 1, password in line 2)
with open('../DS_SpaceTrackCredentials.txt', 'r') as file:
    credentials = file.read().splitlines()

username = credentials[0]
password = credentials[1]

# create a SpaceTrackClient instance
st = SpaceTrackClient(username, password) 

try:
    # Try fetching a single SATCAT entry to verify authentication
    test_data = st.satcat(limit=1, format='json')
    print("Successfully authenticated with Space-Track.org!")
except Exception as e:
    print(f"Failed to authenticate with Space-Track.org: {e}")

Successfully authenticated with Space-Track.org!


In [3]:
# Fetch all SATCAT data from Space-Track
import json
import pandas as pd

satcat_data = st.satcat(format='json') # Fetch all SATCAT data from Space-Track in JSON format
print("Datatype: " + str(type(satcat_data))) # Check the datatype of the SATCAT data
satcat_data = json.loads(satcat_data) # Convert JSON string to a Python list of dictionaries
satcat_df = pd.DataFrame(satcat_data) # Convert list of dictionaries to a Pandas DataFrame
satcat_df

Datatype: <class 'str'>


,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,...,RCSVALUE,RCS_SIZE,FILE,LAUNCH_YEAR,LAUNCH_NUM,LAUNCH_PIECE,CURRENT,OBJECT_NAME,OBJECT_ID,OBJECT_NUMBER
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,...,0,LARGE,1,1957,1,A,Y,SL-1 R/B,1957-001A,1
1,1958-001A,4,PAYLOAD,EXPLORER 1,US,1958-02-01,AFETR,1970-03-31,88.48,33.15,...,0,None,1,1958,1,A,Y,EXPLORER 1,1958-001A,4
2,1958-003A,6,PAYLOAD,EXPLORER 3,US,1958-03-26,AFETR,1958-06-28,103.60,33.50,...,0,None,1,1958,3,A,Y,EXPLORER 3,1958-003A,6
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,...,0,None,1,1958,4,A,Y,SL-1 R/B,1958-004A,7
4,1958-004B,8,PAYLOAD,SPUTNIK 3,CIS,1958-05-15,TTMTR,1960-04-06,88.43,65.06,...,0,LARGE,1,1958,4,B,Y,SPUTNIK 3,1958-004B,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62964,2025-032W,63025,PAYLOAD,STARLINK-32926,US,2025-02-15,AFETR,None,90.72,43.01,...,0,LARGE,9053,2025,32,W,Y,STARLINK-32926,2025-032W,63025
62965,2025-033A,63026,UNKNOWN,OBJECT A,US,2025-02-18,RLLC,None,94.05,59.00,...,0,MEDIUM,9053,2025,33,A,Y,OBJECT A,2025-033A,63026
62966,2025-033C,63028,ROCKET BODY,ELECTRON R/B,US,2025-02-18,RLLC,None,89.98,59.00,...,0,MEDIUM,9053,2025,33,C,Y,ELECTRON R/B,2025-033C,63028
62967,2025-036A,63075,PAYLOAD,CHINASAT-9C,PRC,2025-02-22,XSC,None,631.11,27.63,...,0,None,9053,2025,36,A,Y,CHINASAT-9C,2025-036A,63075


#### 2. Filter Data for only non operating space objects (debris and rocket bodies) that have decayed and have a low earth orbit 

In [4]:
# Check unique values in OBJECT_TYPE column
object_type_counts = satcat_df["OBJECT_TYPE"].value_counts()
print(object_type_counts)

OBJECT_TYPE
DEBRIS         35595
PAYLOAD        19813
ROCKET BODY     6672
UNKNOWN          889
Name: count, dtype: int64


In [5]:
# extract only debris and rocket body objects
debris_df = satcat_df[satcat_df["OBJECT_TYPE"].isin(["DEBRIS", "ROCKET BODY"])]
print("Debris objects: ", debris_df.shape[0])

Debris objects:  42267


In [8]:
# show null values in decay column
print("Not decayed yet: ", debris_df['DECAY'].isnull().sum())

Not decayed yet:  15642


In [7]:
# Filter only decayed objects
decayedDebris_df = debris_df[debris_df["DECAY"].notnull()]
print("Decayed debris objects: ", decayedDebris_df.shape[0])

Decayed debris objects:  26625


In [9]:
# test if apogee is always bigger than perigee to use apogee as criteria for LEO objects
perigee_higher = decayedDebris_df[decayedDebris_df["APOGEE"] < decayedDebris_df["PERIGEE"]]
print(perigee_higher[['APOGEE', 'PERIGEE']])
print(decayedDebris_df[['APOGEE', 'PERIGEE']].dtypes)


      APOGEE PERIGEE
3       1571     206
81       190      93
144     2729      99
278      421      77
597      214      87
...      ...     ...
35753   2076     401
35759   3227     854
36236   2026      97
36687   1036     496
36710   1104     808

[1056 rows x 2 columns]
APOGEE     object
PERIGEE    object
dtype: object


In [ ]:
# Convert APOGEE and PERIGEE columns from type object to numeric types since some where not compared correctly
decayedDebris_df['APOGEE'] = pd.to_numeric(decayedDebris_df['APOGEE'], errors='coerce')
decayedDebris_df['PERIGEE'] = pd.to_numeric(decayedDebris_df['PERIGEE'], errors='coerce')


In [12]:
# test again if apogee is always bigger than perigee
perigee_higher_repeat = (decayedDebris_df["APOGEE"] < decayedDebris_df["PERIGEE"]).all()
print(perigee_higher_repeat)

False


In [13]:
# Filter only objects that are in Low Earth Orbit
# LEO is defined as an orbit with an apogee lower than 2000 km
decayedLEODebris_df = decayedDebris_df[decayedDebris_df["APOGEE"] < 2000].copy()
print("Decayed LEO debris objects: ", decayedLEODebris_df.shape[0])

Decayed LEO debris objects:  25745


#### 3. Delete not needed columns

In [14]:
# perform tests to see which columns can be deleted
print(decayedLEODebris_df['CURRENT'].unique())
print(decayedLEODebris_df['INTLDES'].equals(decayedLEODebris_df['OBJECT_ID']))
print(decayedLEODebris_df['SATNAME'].equals(decayedLEODebris_df['OBJECT_NAME']))


['Y']
True
True


In [15]:
decayedLEODebris_df.drop(columns=['COMMENT', 'COMMENTCODE', 'RCSVALUE', 'FILE', 'LAUNCH_YEAR', 'LAUNCH_NUM', 'LAUNCH_PIECE', 'CURRENT', 'OBJECT_ID', 'OBJECT_NAME', 'OBJECT_NUMBER'], inplace=True)
decayedLEODebris_df

,INTLDES,NORAD_CAT_ID,OBJECT_TYPE,SATNAME,COUNTRY,LAUNCH,SITE,DECAY,PERIOD,INCLINATION,APOGEE,PERIGEE,RCS_SIZE
0,1957-001A,1,ROCKET BODY,SL-1 R/B,CIS,1957-10-04,TTMTR,1957-12-01,96.19,65.10,938.0,214.0,LARGE
3,1958-004A,7,ROCKET BODY,SL-1 R/B,CIS,1958-05-15,TTMTR,1958-12-03,102.74,65.14,1571.0,206.0,None
17,1960-003A,30,ROCKET BODY,THOR ABLESTAR R/B,US,1960-04-13,AFETR,1961-08-18,88.38,51.25,194.0,194.0,None
20,1960-003C,33,DEBRIS,THOR ABLESTAR DEB,US,1960-04-13,AFETR,1960-07-17,93.59,51.29,615.0,285.0,None
22,1960-005B,35,ROCKET BODY,SL-3 R/B,CIS,1960-05-15,TTMTR,1960-07-17,88.48,64.89,199.0,199.0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...
50795,1977-065GN,39101,DEBRIS,DELTA 1 DEB,US,1977-07-14,AFETR,2025-02-21,89.81,29.28,286.0,244.0,SMALL
51252,2000-055RC,42360,DEBRIS,NOAA 16 DEB,US,2000-09-21,AFWTR,2025-02-21,92.35,98.81,404.0,375.0,SMALL
54311,2024-140SU,61283,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2025-02-21,90.32,89.18,305.0,275.0,SMALL
54462,2024-140ADL,61844,DEBRIS,CZ-6A DEB,PRC,2024-08-06,TSC,2025-02-21,90.44,89.24,310.0,282.0,SMALL


#### 4. Inspect filtered data

In [16]:
# get basic statistics of numerical columns
print(decayedLEODebris_df.describe())

             APOGEE       PERIGEE
count  25745.000000  25745.000000
mean     351.663935    255.018877
std      240.543548    130.488593
min        0.000000      0.000000
25%      206.000000    162.000000
50%      283.000000    212.000000
75%      404.000000    315.000000
max     1998.000000   1478.000000


In [17]:
# Check for missing values
print(decayedLEODebris_df.isnull().sum())

INTLDES            0
NORAD_CAT_ID       0
OBJECT_TYPE        0
SATNAME            0
COUNTRY            0
LAUNCH             0
SITE               0
DECAY              0
PERIOD             0
INCLINATION        0
APOGEE             0
PERIGEE            0
RCS_SIZE        6895
dtype: int64


##### RCS_SIZE should later be dropped before the data is used for training the model since it has too many missing values.

In [18]:
# Data distribution
print(decayedLEODebris_df["OBJECT_TYPE"].value_counts())  # Number of debris vs. rocket bodies
print(decayedLEODebris_df["COUNTRY"].value_counts().head(10))  # Top 10 countries with most debris
print(decayedLEODebris_df["RCS_SIZE"].value_counts(dropna=False))  # Radar Cross Section size

OBJECT_TYPE
DEBRIS         21749
ROCKET BODY     3996
Name: count, dtype: int64
COUNTRY
CIS     15517
US       6061
PRC      2230
FR        740
IND       515
JPN       413
ISS       100
CHBZ       75
ESA        31
ORB        27
Name: count, dtype: int64
RCS_SIZE
SMALL     12090
None       6895
MEDIUM     3707
LARGE      3053
Name: count, dtype: int64


In [19]:
# save to csv
decayedLEODebris_df.to_csv('../Data/DS_DecayedLEODebris.csv', index=False)
decayedLEODebris_df['NORAD_CAT_ID'].to_csv('../Data/DS_NoradIDs.csv', index=False)